# Exploratory Analysis and Model Training\nThis notebook performs exploratory data analysis (EDA) on the BACE dataset and trains machine learning models to predict biological activity.\nThe dataset used is the BACE dataset (from Kaggle or similar sources) containing molecules with binary activity labels (active/inactive against BACE-1).

In [ ]:
# Import necessary libraries\nimport pandas as pd\nimport numpy as np\nimport matplotlib.pyplot as plt\nimport seaborn as sns\nimport os\nimport warnings\nwarnings.filterwarnings('ignore')\n\n# Set plotting style\nsns.set_style('whitegrid')\nplt.rcParams['figure.figsize'] = (10, 6)

## Loading the Data\nWe'll load the processed data that includes molecular descriptors and features.

In [ ]:
# Load the processed features\nfeatures_path = 'data/processed/bace_features.csv'\nif os.path.exists(features_path):\n    df = pd.read_csv(features_path)\n    print(f'Loaded features data shape: {df.shape}')\n    df.head()\nelse:\n    print(f'File not found: {features_path}')\n    print('Please run the data processing scripts first.')

## Exploratory Data Analysis\nLet's examine the dataset, check for missing values, and visualize feature distributions.

In [ ]:
# Basic info\nprint('Dataset shape:', df.shape)\nprint('\nColumn names:')\nfor col in df.columns:\n    print(f'  {col}')\nprint('\nData types:')\nprint(df.dtypes)\nprint('\nMissing values:')\nprint(df.isnull().sum())

In [ ]:
# Check the target variable distribution\nif 'Class' in df.columns:\n    class_counts = df['Class'].value_counts()\n    print('Class distribution:')\n    print(class_counts)\n    print(f'\nPercentage of active compounds: {class_counts[1]/len(df)*100:.2f}%')\n    # Plot class distribution\n    plt.figure()\n    sns.countplot(x='Class', data=df)\n    plt.title('Distribution of Active vs Inactive Compounds')\n    plt.xlabel('Class (0: Inactive, 1: Active)')\n    plt.ylabel('Count')\n    plt.show()\nelse:\n    print('Target column \'Class\' not found.')

In [ ]:
# Lipinski descriptors statistics\nlipinski_cols = [col for col in df.columns if col.startswith('lipinski_')]\nif lipinski_cols:\n    print('Lipinski descriptors:')\n    print(df[lipinski_cols].describe())\n    # Plot distributions\n    df[lipinski_cols].hist(bins=20, figsize=(12, 8))\n    plt.suptitle('Distributions of Lipinski Descriptors')\n    plt.tight_layout()\n    plt.show()\nelse:\n    print('No Lipinski descriptor columns found.')

In [ ]:
# Check Lipinski pass rate\nif 'lipinski_pass' in df.columns:\n    pass_rate = df['lipinski_pass'].mean()\n    print(f'Percentage of compounds passing Lipinski rule: {pass_rate*100:.2f}%')\n    # Plot pass rate by class\n    if 'Class' in df.columns:\n        pass_by_class = df.groupby('Class')['lipinski_pass'].mean()\n        print('Lipinski pass rate by class:')\n        print(pass_by_class)\n        plt.figure()\n        pass_by_class.plot(kind='bar')\n        plt.title('Lipinski Pass Rate by Activity Class')\n        plt.xlabel('Class (0: Inactive, 1: Active)')\n        plt.ylabel('Proportion Passing Lipinski')\n        plt.xticks(rotation=0)\n        plt.show()\nelse:\n    print('Lipinski pass column not found.')

## Feature Preparation for Modeling\nWe'll prepare the feature matrix and target vector for model training, similar to what's done in `src/prepare_data.py`.

In [ ]:
# Prepare features and target\n# We'll exclude the SMILES column and the target column, and use all other features\nif 'smiles' in df.columns and 'Class' in df.columns:\n    X = df.drop(columns=['smiles', 'Class'])\n    y = df['Class'].values\n    print(f'Feature matrix shape: {X.shape}')\n    print(f'Target vector shape: {y.shape}')\n    print(f'Feature names: {list(X.columns)}')\nelse:\n    print('Required columns not found.')\n    X = None\n    y = None

## Model Training and Evaluation\nWe'll train the same models as in `src/train_models.py`: Random Forest, Gradient Boosting, and a Neural Network (MLP).\nWe'll use a train-test split and evaluate performance using ROC-AUC, accuracy, F1-score, precision, and recall.

In [ ]:
# If we have the data, proceed with modeling\nif X is not None and y is not None:\n    from sklearn.model_selection import train_test_split\n    from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier\n    from sklearn.neural_network import MLPClassifier\n    from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, roc_curve, precision_recall_curve\n    import itertools\n    \n    # Split the data\n    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)\n    print(f'Training set size: {X_train.shape}')\n    print(f'Test set size: {X_test.shape}')\n    \n    # Define models\n    models = {\n        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),\n        'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),\n        'Neural Network': MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=500, random_state=42)\n    }\n    \n    # Train and evaluate each model\n    results = {}\n    for name, model in models.items():\n        print(f'\nTraining {name}...')\n        model.fit(X_train, y_train)\n        \n        # Predictions\n        y_pred = model.predict(X_test)\n        y_pred_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else model.decision_function(X_test)\n        \n        # Metrics\n        roc_auc = roc_auc_score(y_test, y_pred_proba)\n        accuracy = accuracy_score(y_test, y_pred)\n        f1 = f1_score(y_test, y_pred)\n        precision = precision_score(y_test, y_pred)\n        recall = recall_score(y_test, y_pred)\n        \n        results[name] = {\n            'ROC-AUC': roc_auc,\n            'Accuracy': accuracy,\n            'F1-Score': f1,\n            'Precision': precision,\n            'Recall': recall\n        }\n        \n        print(f'{name} - ROC-AUC: {roc_auc:.4f}, Accuracy: {accuracy:.4f}, F1: {f1:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}')\n    \n    # Convert results to DataFrame for easy viewing\n    results_df = pd.DataFrame(results).T\n    print('\n=== Model Performance Summary ===')\n    display(results_df)\n    \n    # Plot ROC curves\n    plt.figure()\n    for name, model in models.items():\n        y_pred_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else model.decision_function(X_test)\n        fpr, tpr, _ = roc_curve(y_test, y_pred_proba)\n        plt.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc_score(y_test, y_pred_proba):.3f})')\n    plt.plot([0, 1], [0, 1], 'k--', label='Random Guess')\n    plt.xlabel('False Positive Rate')\n    plt.ylabel('True Positive Rate')\n    plt.title('ROC Curves')\n    plt.legend(loc='lower right')\n    plt.show()\n    \n    # Plot precision-recall curves\n    plt.figure()\n    for name, model in models.items():\n        y_pred_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else model.decision_function(X_test)\n        precision, recall, _ = precision_recall_curve(y_test, y_pred_proba)\n        plt.plot(recall, precision, label=f'{name}')\n    plt.xlabel('Recall')\n    plt.ylabel('Precision')\n    plt.title('Precision-Recall Curves')\n    plt.legend(loc='lower left')\n    plt.show()\n    \n    # Plot confusion matrices\n    fig, axes = plt.subplots(1, 3, figsize=(15, 5))\n    for idx, (name, model) in enumerate(models.items()):\n        y_pred = model.predict(X_test)\n        cm = confusion_matrix(y_test, y_pred)\n        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx])\n        axes[idx].set_title(f'Confusion Matrix - {name}')\n        axes[idx].set_xlabel('Predicted')\n        axes[idx].set_ylabel('Actual')\n    plt.tight_layout()\n    plt.show()\nelse:\n    print('Data not prepared. Skipping modeling.')

## Conclusion\nIn this notebook, we:\n1. Loaded the processed BACE dataset with molecular descriptors.\n2. Performed exploratory data analysis, including class distribution and Lipinski descriptor statistics.\n3. Prepared the feature matrix and target vector for modeling.\n4. Trained and evaluated three machine learning models: Random Forest, Gradient Boosting, and a Neural Network.\n5. Visualized model performance using ROC curves, precision-recall curves, and confusion matrices.\n\nBased on the results, we can select the best-performing model for drug candidate prediction.\nThe Random Forest model often provides a good balance of metrics for this type of classification task.\n\nNext steps could include hyperparameter tuning, feature selection, or trying more advanced models.